# Appendix 11. Model Metrics 기초

## 1. Goal

이 notebook은 metric을 단순한 숫자 목록이 아니라 **target의 종류, prediction의 형태, 운영 의사결정**을 연결하는 평가 contract로 다룹니다. Binary classification, multiclass classification, numeric target을 예측하는 regression을 같은 흐름으로 비교합니다.

학습 목표는 다음과 같습니다.

- `y_true`, class label, score·probability, numeric prediction의 차이를 구분합니다.
- label metric, ranking metric, probability metric과 regression loss가 답하는 질문을 설명합니다.
- class imbalance, threshold, averaging, target scale과 outlier가 metric에 미치는 영향을 확인합니다.
- primary metric 하나만 보지 않고 baseline, guardrail, slice와 uncertainty를 함께 기록합니다.

모든 예제는 고정 seed로 만든 합성 validation data를 사용합니다. 여기서 얻은 수치는 실제 PhysioNet model의 성능이나 release evidence가 아닙니다.

## 2. Setup

`scikit-learn.metrics`의 함수는 이미 만들어진 `y_true`와 prediction을 평가합니다. 이 notebook은 model fitting보다 metric의 입력과 해석에 집중합니다. Chart는 앞에서 익힌 Matplotlib의 Figure와 Axes API로 구성합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    explained_variance_score,
    f1_score,
    log_loss,
    matthews_corrcoef,
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_pinball_loss,
    mean_squared_error,
    median_absolute_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    root_mean_squared_error,
    root_mean_squared_log_error,
    top_k_accuracy_score,
)

SEED = 20260714
rng = np.random.default_rng(SEED)

COLORS = {
    "blue": "#2F6B9A",
    "blue_light": "#A9C6DC",
    "gold": "#C6922D",
    "orange": "#D66A3A",
    "ink": "#27313A",
    "gray": "#8A949E",
    "grid": "#E5E9ED",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": COLORS["ink"],
    "axes.labelcolor": COLORS["ink"],
    "axes.titlecolor": COLORS["ink"],
    "axes.grid": True,
    "grid.color": COLORS["grid"],
    "grid.linewidth": 0.8,
    "font.size": 10,
})

pd.set_option("display.max_columns", 20)

## 3. Steps

### 3-1. Metric contract를 먼저 정의하기

#### 3-1-1. task와 prediction 입력 형태

Metric을 고르기 전에 target과 prediction이 무엇인지 확인합니다. Binary probability는 보통 `(n_samples,)`, multiclass probability는 class마다 한 열을 가지는 `(n_samples, n_classes)`, regression prediction은 연속값 `(n_samples,)`입니다. 같은 이름의 metric이라도 label, probability, score 중 무엇을 입력하는지 다를 수 있습니다.

In [ ]:
prediction_contract = pd.DataFrame(
    [
        {
            "task": "binary classification",
            "target": "0/1 class label",
            "prediction": "positive-class score or probability",
            "typical_shape": "(n_samples,)",
        },
        {
            "task": "multiclass classification",
            "target": "one class per sample",
            "prediction": "one probability per class",
            "typical_shape": "(n_samples, n_classes)",
        },
        {
            "task": "regression",
            "target": "numeric value",
            "prediction": "numeric estimate",
            "typical_shape": "(n_samples,)",
        },
    ]
)
prediction_contract

#### 3-1-2. score, loss와 utility를 구분하기

Score는 보통 클수록 좋고 loss는 작을수록 좋습니다. 그러나 이름만 보고 방향을 추측하면 안 됩니다. 예를 들어 ROC AUC와 $R^2$는 클수록 좋지만 log loss, MAE와 RMSE는 작을수록 좋습니다. scikit-learn의 model selection `scoring=`에서는 모든 scorer를 “클수록 좋다”는 규칙으로 맞추기 위해 loss에 `neg_` prefix를 붙입니다. 보고서에는 다시 원래 단위와 방향으로 변환해 기록하는 편이 읽기 쉽습니다.

In [ ]:
metric_families = pd.DataFrame(
    [
        ("label decision", "accuracy / recall / F1", "predicted label", "higher"),
        ("ranking", "ROC AUC / Average Precision", "score or probability", "higher"),
        ("probability", "log loss / Brier loss", "probability", "lower"),
        ("numeric error", "MAE / RMSE / pinball loss", "numeric prediction", "lower"),
        ("relative fit", "R² / explained variance", "numeric prediction", "higher"),
    ],
    columns=["family", "examples", "required_prediction", "better_direction"],
)
metric_families

#### 3-1-3. primary, guardrail, slice와 baseline

Primary metric은 model 선택의 주 기준입니다. Guardrail은 primary가 좋아져도 놓치면 안 되는 위험을 감시합니다. Slice metric은 cohort나 class별 실패를 드러냅니다. 마지막으로 같은 split에서 계산한 단순 baseline이 있어야 model 수치가 실제 개선인지 판단할 수 있습니다.

Metric contract에는 계산식만이 아니라 target revision, evaluation split, prediction column, positive label 또는 class order, threshold, averaging, sample weight, 방향과 허용 기준을 함께 고정해야 합니다.

In [ ]:
metric_contract_template = pd.DataFrame(
    [
        ("primary", "valid_auroc", "binary", "probability", "higher", "overall"),
        ("guardrail", "valid_recall_at_threshold", "binary", "label", "higher", "overall"),
        ("slice", "valid_macro_f1", "multiclass", "label", "higher", "per class + macro"),
        ("primary", "valid_mae_hours", "regression", "numeric", "lower", "overall + cohort"),
    ],
    columns=["role", "metric_key", "task", "prediction_input", "direction", "aggregation"],
)
metric_contract_template

### 3-2. Binary classification metric

#### 3-2-1. label, probability와 threshold

Binary classifier는 positive class의 score 또는 probability를 만들고, threshold를 적용해 최종 label을 정합니다. Probability metric은 threshold 없이 계산하지만 confusion matrix, precision, recall과 F1은 선택한 threshold에 따라 달라집니다. Threshold는 validation evidence와 운영 비용을 기준으로 정하고 sealed test 결과에 맞춰 조정하지 않습니다.

In [ ]:
n_binary = 420
binary_risk = rng.normal(size=n_binary)
binary_event_probability = 1 / (1 + np.exp(-(-1.35 + 1.25 * binary_risk)))
y_binary = rng.binomial(1, binary_event_probability)

# 완벽하지 않은 model을 흉내 내기 위해 signal과 noise를 섞습니다.
binary_score = 0.95 * binary_risk + rng.normal(scale=0.9, size=n_binary)
p_binary = 1 / (1 + np.exp(-(-1.1 + binary_score)))
binary_threshold = 0.40
y_binary_pred = (p_binary >= binary_threshold).astype(int)

binary_preview = pd.DataFrame({
    "y_true": y_binary,
    "positive_probability": p_binary,
    "predicted_label": y_binary_pred,
})

print({
    "samples": n_binary,
    "positive_rate": y_binary.mean(),
    "alert_rate": y_binary_pred.mean(),
    "threshold": binary_threshold,
})
binary_preview.head()

#### 3-2-2. confusion matrix와 label metric

Confusion matrix는 실제 class와 예측 class의 조합을 count합니다. Precision은 positive라고 알린 sample 중 실제 positive의 비율, recall(sensitivity)은 실제 positive 중 찾아낸 비율입니다. Specificity는 실제 negative 중 negative로 남긴 비율입니다. F1은 precision과 recall의 조화평균이고, balanced accuracy는 class별 recall의 평균입니다.

Accuracy는 직관적이지만 class imbalance가 크면 majority class만 예측해도 높을 수 있습니다. 각 metric의 분모와 실제 운영 비용을 먼저 확인합니다.

In [ ]:
binary_confusion = confusion_matrix(y_binary, y_binary_pred, labels=[0, 1])
true_negative, false_positive, false_negative, true_positive = binary_confusion.ravel()

binary_label_metrics = pd.Series({
    "accuracy": accuracy_score(y_binary, y_binary_pred),
    "balanced_accuracy": balanced_accuracy_score(y_binary, y_binary_pred),
    "precision": precision_score(y_binary, y_binary_pred, zero_division=0),
    "recall_sensitivity": recall_score(y_binary, y_binary_pred),
    "specificity": true_negative / (true_negative + false_positive),
    "f1": f1_score(y_binary, y_binary_pred),
    "matthews_corrcoef": matthews_corrcoef(y_binary, y_binary_pred),
})

print(pd.DataFrame(
    binary_confusion,
    index=["actual_negative", "actual_positive"],
    columns=["pred_negative", "pred_positive"],
))
binary_label_metrics.to_frame("value")

#### 3-2-3. threshold에 따른 trade-off

Threshold를 낮추면 일반적으로 더 많은 sample을 positive로 분류해 recall과 alert rate가 올라갑니다. 반대로 false positive가 늘면서 precision이나 specificity가 낮아질 수 있습니다. 하나의 F1 최대값만 찾기보다 운영 가능한 alert volume, false negative 비용과 capacity를 함께 검토합니다.

In [ ]:
threshold_rows = []
for threshold in np.linspace(0.10, 0.90, 17):
    predicted = (p_binary >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_binary, predicted, labels=[0, 1]).ravel()
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_binary, predicted, zero_division=0),
        "recall": recall_score(y_binary, predicted),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y_binary, predicted, zero_division=0),
        "alert_rate": predicted.mean(),
    })

binary_threshold_metrics = pd.DataFrame(threshold_rows)
binary_threshold_metrics.loc[
    (binary_threshold_metrics["threshold"] - binary_threshold).abs().idxmin()
].to_frame("selected_threshold")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8), layout="constrained")
for column, color, linestyle in [
    ("precision", COLORS["blue"], "-"),
    ("recall", COLORS["orange"], "-"),
    ("specificity", COLORS["gold"], "--"),
    ("alert_rate", COLORS["gray"], ":"),
]:
    ax.plot(
        binary_threshold_metrics["threshold"],
        binary_threshold_metrics[column],
        label=column,
        color=color,
        linestyle=linestyle,
        linewidth=2,
    )

ax.axvline(binary_threshold, color=COLORS["ink"], linewidth=1.2, label="selected threshold")
ax.set(
    title=f"Binary metrics by decision threshold (validation n={n_binary})",
    xlabel="threshold",
    ylabel="metric value / prediction share",
    xlim=(0.1, 0.9),
    ylim=(0, 1.02),
)
ax.legend(
    ncol=5,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.16),
)
plt.show()

#### 3-2-4. ranking metric: AUROC, PR-AUC와 Average Precision

AUROC는 ROC curve 아래 면적이며, 임의의 positive sample이 임의의 negative sample보다 높은 score를 받을 가능성으로 해석할 수 있습니다. `roc_auc_score`로 직접 계산한 값과 `roc_curve`의 점에 `auc`를 적용한 값이 같은지 확인합니다. 무작위 ranking의 AUROC 기준은 0.5입니다.

PR-AUC는 precision-recall curve 아래 면적을 trapezoid 방식으로 적분한 값입니다. Average Precision(AP)도 같은 curve를 요약하지만 recall 증가분으로 precision을 가중하는 step 방식이므로 일반적인 trapezoidal PR-AUC와 정확히 같은 값은 아닙니다. Positive class가 희소할 때 둘 다 prevalence와 함께 해석합니다.

AUROC, PR-AUC와 AP는 모두 ranking을 평가하므로 probability가 실제 risk와 맞는지, 특정 threshold에서 운영 결과가 어떤지는 별도로 확인해야 합니다.

In [ ]:
false_positive_rate, true_positive_rate, _ = roc_curve(y_binary, p_binary)
curve_precision, curve_recall, _ = precision_recall_curve(y_binary, p_binary)

binary_ranking_metrics = pd.Series({
    "auroc": roc_auc_score(y_binary, p_binary),
    "auroc_from_curve": auc(false_positive_rate, true_positive_rate),
    "pr_auc_trapezoid": auc(curve_recall, curve_precision),
    "average_precision": average_precision_score(y_binary, p_binary),
    "positive_prevalence": y_binary.mean(),
})
binary_baseline_metrics = pd.Series({
    "random_ranking_auroc": 0.5,
    "prevalence_ap_reference": y_binary.mean(),
    "all_negative_accuracy": accuracy_score(
        y_binary,
        np.zeros_like(y_binary),
    ),
})

display(binary_ranking_metrics.to_frame("model_value"))
binary_baseline_metrics.to_frame("reference_value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), layout="constrained")

axes[0].plot(false_positive_rate, true_positive_rate, color=COLORS["blue"], linewidth=2)
axes[0].plot([0, 1], [0, 1], color=COLORS["gray"], linestyle="--", linewidth=1)
axes[0].set(
    title=f"ROC curve (AUROC={binary_ranking_metrics['auroc']:.3f})",
    xlabel="false positive rate",
    ylabel="true positive rate",
    xlim=(0, 1),
    ylim=(0, 1),
)

axes[1].plot(curve_recall, curve_precision, color=COLORS["orange"], linewidth=2)
axes[1].axhline(
    y_binary.mean(),
    color=COLORS["gray"],
    linestyle="--",
    linewidth=1,
    label="positive prevalence",
)
axes[1].set(
    title=(
        f"Precision-recall curve "
        f"(PR-AUC={binary_ranking_metrics['pr_auc_trapezoid']:.3f}, "
        f"AP={binary_ranking_metrics['average_precision']:.3f})"
    ),
    xlabel="recall",
    ylabel="precision",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[1].legend(frameon=False)
plt.show()

#### 3-2-5. probability metric과 calibration

Log loss는 실제 class에 낮은 probability를 준 confident error를 강하게 벌점합니다. Brier loss는 binary outcome과 positive probability의 squared error입니다. 둘 다 작을수록 좋지만 discrimination과 calibration을 동시에 반영하므로 loss 하나만으로 calibration이 좋아졌다고 단정할 수 없습니다.

Reliability diagram은 probability bin별 평균 예측값과 실제 positive 비율을 비교합니다. Data가 적은데 bin을 지나치게 많이 만들면 각 점이 불안정해집니다. 여기서는 probability 분포를 함께 표시해 bin별 sample 밀도를 확인합니다.

In [ ]:
binary_probability_metrics = pd.Series({
    "log_loss": log_loss(y_binary, p_binary),
    "brier_loss": brier_score_loss(y_binary, p_binary),
})

fraction_positive, mean_predicted_probability = calibration_curve(
    y_binary,
    p_binary,
    n_bins=8,
    strategy="quantile",
)
binary_probability_metrics.to_frame("value")

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(10, 4.3),
    layout="constrained",
    gridspec_kw={"width_ratios": [1.15, 1]},
)

axes[0].plot([0, 1], [0, 1], color=COLORS["gray"], linestyle="--", label="ideal")
axes[0].plot(
    mean_predicted_probability,
    fraction_positive,
    color=COLORS["blue"],
    marker="o",
    linewidth=2,
    label="model bins",
)
axes[0].set(
    title="Binary reliability diagram",
    xlabel="mean predicted probability",
    ylabel="observed positive fraction",
    xlim=(0, 1),
    ylim=(0, 1),
)
axes[0].legend(frameon=False)

axes[1].hist(p_binary, bins=np.linspace(0, 1, 16), color=COLORS["blue_light"], edgecolor=COLORS["blue"])
axes[1].set(
    title="Predicted probability distribution",
    xlabel="positive-class probability",
    ylabel="samples",
    xlim=(0, 1),
)
plt.show()

### 3-3. Multiclass classification metric

#### 3-3-1. multiclass와 multilabel을 구분하기

Multiclass는 sample 하나가 여러 class 중 정확히 하나에 속합니다. 따라서 `y_true`는 class label 한 개이고 `predict_proba`의 한 행은 class별 probability이며 합이 1입니다. Multilabel은 sample 하나가 여러 label을 동시에 가질 수 있어 별도의 binary indicator matrix와 다른 averaging 해석이 필요합니다. 이 notebook은 multiclass에 집중합니다.

Probability 열 순서는 estimator의 `classes_`와 같아야 합니다. Class name을 별도로 보관하고 shape와 row sum을 먼저 점검합니다.

In [ ]:
class_labels = np.array(["low", "moderate", "severe"])
n_multiclass = 480
class_prevalence = np.array([0.58, 0.29, 0.13])
y_multiclass = rng.choice(class_labels, size=n_multiclass, p=class_prevalence)

class_to_index = {label: index for index, label in enumerate(class_labels)}
true_class_index = np.array([class_to_index[label] for label in y_multiclass])
multiclass_logits = rng.normal(scale=1.05, size=(n_multiclass, len(class_labels)))
multiclass_logits[np.arange(n_multiclass), true_class_index] += 1.75
multiclass_logits[:, 0] += 0.22  # majority class 쪽으로 약한 bias를 둡니다.

exp_logits = np.exp(multiclass_logits - multiclass_logits.max(axis=1, keepdims=True))
p_multiclass = exp_logits / exp_logits.sum(axis=1, keepdims=True)
y_multiclass_pred = class_labels[p_multiclass.argmax(axis=1)]

print({
    "target_shape": y_multiclass.shape,
    "probability_shape": p_multiclass.shape,
    "class_order": class_labels.tolist(),
    "row_sum_range": (p_multiclass.sum(axis=1).min(), p_multiclass.sum(axis=1).max()),
})
pd.Series(y_multiclass, name="actual_class").value_counts(normalize=True).rename("share")

#### 3-3-2. multiclass confusion matrix

Multiclass confusion matrix의 행은 actual class, 열은 predicted class입니다. Count matrix는 오류 volume을 보여주고, 행 기준 normalize는 class마다 어디로 잘못 분류되는지 비율을 보여줍니다. Class order를 명시하지 않으면 표와 probability 열의 순서를 잘못 연결할 수 있습니다.

In [ ]:
multiclass_confusion_count = confusion_matrix(
    y_multiclass,
    y_multiclass_pred,
    labels=class_labels,
)
multiclass_confusion_rate = confusion_matrix(
    y_multiclass,
    y_multiclass_pred,
    labels=class_labels,
    normalize="true",
)

pd.DataFrame(
    multiclass_confusion_count,
    index=[f"actual_{label}" for label in class_labels],
    columns=[f"pred_{label}" for label in class_labels],
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5), layout="constrained")
image = ax.imshow(multiclass_confusion_rate, vmin=0, vmax=1, cmap="Blues")

for row in range(len(class_labels)):
    for column in range(len(class_labels)):
        value = multiclass_confusion_rate[row, column]
        ax.text(
            column,
            row,
            f"{value:.1%}\n(n={multiclass_confusion_count[row, column]})",
            ha="center",
            va="center",
            color="white" if value > 0.55 else COLORS["ink"],
        )

ax.grid(False)
ax.set(
    title="Multiclass confusion matrix (row-normalized)",
    xlabel="predicted class",
    ylabel="actual class",
    xticks=np.arange(len(class_labels)),
    yticks=np.arange(len(class_labels)),
    xticklabels=class_labels,
    yticklabels=class_labels,
)
fig.colorbar(image, ax=ax, label="share within actual class", shrink=0.8)
plt.show()

#### 3-3-3. per-class metric과 averaging

Multiclass precision·recall·F1은 class 하나를 positive, 나머지를 negative로 보는 one-vs-rest 문제로 계산할 수 있습니다. `average=None`은 class별 값을 그대로 반환합니다.

- `macro`: class별 metric을 같은 비중으로 평균냅니다. Rare class도 똑같이 중요할 때 유용합니다.
- `weighted`: 각 class의 support로 가중합니다. 전체 sample 구성은 반영하지만 rare class의 실패가 작게 보일 수 있습니다.
- `micro`: 모든 class의 TP·FP·FN을 합친 뒤 계산합니다. Single-label multiclass에서 모든 class를 포함하면 micro precision·recall·F1은 accuracy와 같습니다.

평균만 보고 끝내지 말고 반드시 per-class support와 metric을 함께 확인합니다.

In [ ]:
multiclass_report = pd.DataFrame(
    classification_report(
        y_multiclass,
        y_multiclass_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
).T

multiclass_summary = pd.Series({
    "accuracy": accuracy_score(y_multiclass, y_multiclass_pred),
    "balanced_accuracy": balanced_accuracy_score(y_multiclass, y_multiclass_pred),
    "f1_macro": f1_score(y_multiclass, y_multiclass_pred, average="macro"),
    "f1_weighted": f1_score(y_multiclass, y_multiclass_pred, average="weighted"),
    "f1_micro": f1_score(y_multiclass, y_multiclass_pred, average="micro"),
    "matthews_corrcoef": matthews_corrcoef(y_multiclass, y_multiclass_pred),
})
multiclass_majority_label = pd.Series(y_multiclass).mode().iat[0]
y_multiclass_baseline = np.repeat(multiclass_majority_label, n_multiclass)
multiclass_baseline_metrics = pd.Series({
    "majority_class": multiclass_majority_label,
    "majority_accuracy": accuracy_score(y_multiclass, y_multiclass_baseline),
    "majority_macro_f1": f1_score(
        y_multiclass,
        y_multiclass_baseline,
        average="macro",
    ),
})

display(multiclass_report.loc[class_labels])
display(multiclass_summary.to_frame("model_value"))
multiclass_baseline_metrics.to_frame("reference_value")

In [ ]:
per_class_plot = multiclass_report.loc[class_labels, ["precision", "recall", "f1-score"]]
ax = per_class_plot.plot.bar(
    figsize=(8.5, 4.8),
    color=[COLORS["blue"], COLORS["orange"], COLORS["gold"]],
    edgecolor=COLORS["ink"],
    width=0.75,
)
ax.set(
    title="Precision, recall and F1 by class",
    xlabel="class",
    ylabel="metric value",
    ylim=(0, 1),
)
ax.tick_params(axis="x", rotation=0)
ax.legend(frameon=False, ncol=3, loc="lower center")
plt.show()

#### 3-3-4. multiclass probability와 ranking metric

Multiclass log loss는 실제 class에 부여한 probability를 평가합니다. `top_k_accuracy_score`는 정답 class가 상위 k개 probability 안에 있는지를 봅니다. 후보를 몇 개 제시하는 검색·triage workflow에는 의미가 있지만 최종 class 하나를 결정하는 문제에서는 top-1 결과를 대신할 수 없습니다.

Multiclass ROC AUC는 one-vs-rest(`ovr`) 또는 one-vs-one(`ovo`)으로 확장합니다. `macro`와 `weighted` averaging의 의미가 다르므로 method와 average를 metric key에 명시합니다. 또한 class별 OvR AUC를 확인해 평균이 가리는 실패를 찾습니다.

In [ ]:
multiclass_probability_metrics = pd.Series({
    "log_loss": log_loss(y_multiclass, p_multiclass, labels=class_labels),
    "top_2_accuracy": top_k_accuracy_score(
        y_multiclass,
        p_multiclass,
        k=2,
        labels=class_labels,
    ),
    "roc_auc_ovr_macro": roc_auc_score(
        y_multiclass,
        p_multiclass,
        labels=class_labels,
        multi_class="ovr",
        average="macro",
    ),
    "roc_auc_ovr_weighted": roc_auc_score(
        y_multiclass,
        p_multiclass,
        labels=class_labels,
        multi_class="ovr",
        average="weighted",
    ),
    "roc_auc_ovo_macro": roc_auc_score(
        y_multiclass,
        p_multiclass,
        labels=class_labels,
        multi_class="ovo",
        average="macro",
    ),
})

per_class_auc = pd.Series(
    roc_auc_score(
        y_multiclass,
        p_multiclass,
        labels=class_labels,
        multi_class="ovr",
        average=None,
    ),
    index=class_labels,
    name="ovr_roc_auc",
)

display(multiclass_probability_metrics.to_frame("value"))
per_class_auc.to_frame()

#### 3-3-5. class별 one-vs-rest calibration

Multiclass probability도 class별로 “해당 class 대 나머지” binary 문제로 바꾸어 reliability를 점검할 수 있습니다. 한 sample의 class probability가 서로 합쳐 1이므로 각 curve가 독립적이지 않다는 점에 주의합니다. Rare class는 bin별 sample이 적어 curve가 더 불안정할 수 있으므로 support와 probability 분포를 함께 봅니다.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.2), layout="constrained")
ax.plot([0, 1], [0, 1], color=COLORS["gray"], linestyle="--", label="ideal")

class_colors = [COLORS["blue"], COLORS["orange"], COLORS["gold"]]
for class_index, (class_label, color) in enumerate(zip(class_labels, class_colors)):
    binary_target_for_class = (y_multiclass == class_label).astype(int)
    fraction, mean_probability = calibration_curve(
        binary_target_for_class,
        p_multiclass[:, class_index],
        n_bins=6,
        strategy="quantile",
    )
    ax.plot(
        mean_probability,
        fraction,
        marker="o",
        linewidth=2,
        color=color,
        label=f"{class_label} (n={binary_target_for_class.sum()})",
    )

ax.set(
    title="One-vs-rest reliability by class",
    xlabel="mean predicted class probability",
    ylabel="observed class fraction",
    xlim=(0, 1),
    ylim=(0, 1),
)
ax.legend(frameon=False)
plt.show()

#### 3-3-6. confidence와 error slice

Max probability가 높다고 항상 정답은 아닙니다. Confidence band별 accuracy와 sample 수를 함께 보면 confident error가 어디에 모이는지 확인할 수 있습니다. 이 분석은 calibration을 대신하지 않지만 review queue나 abstention policy를 설계할 때 유용한 error slice가 됩니다.

In [ ]:
multiclass_error_frame = pd.DataFrame({
    "actual": y_multiclass,
    "predicted": y_multiclass_pred,
    "confidence": p_multiclass.max(axis=1),
})
multiclass_error_frame["correct"] = (
    multiclass_error_frame["actual"] == multiclass_error_frame["predicted"]
)
multiclass_error_frame["confidence_band"] = pd.cut(
    multiclass_error_frame["confidence"],
    bins=[0.0, 0.5, 0.7, 0.85, 1.0],
    include_lowest=True,
)

confidence_slice = (
    multiclass_error_frame.groupby("confidence_band", observed=False)
    .agg(samples=("correct", "size"), accuracy=("correct", "mean"))
)
confidence_slice

### 3-4. Numeric target와 regression metric

#### 3-4-1. residual과 baseline

Regression은 class가 아니라 연속적인 numeric target을 예측합니다. Residual은 보통 `y_true - y_pred`로 정의합니다. 양수 residual은 model이 실제값보다 낮게 예측했다는 뜻입니다.

Mean 또는 median으로 모든 sample을 예측하는 단순 baseline과 비교해야 model이 target의 변동을 설명하는지 알 수 있습니다. MAE를 primary로 쓰면 median baseline, squared error를 쓰면 mean baseline이 자연스러운 기준입니다.

In [ ]:
n_regression = 420
regression_severity = rng.normal(size=n_regression)
y_regression = np.exp(2.1 + 0.42 * regression_severity + rng.normal(scale=0.28, size=n_regression))

# 큰 target에서 error spread가 커지고, 높은 severity를 조금 낮게 예측하는 상황입니다.
regression_noise = rng.normal(scale=1.2 + 0.08 * y_regression, size=n_regression)
y_regression_pred = np.clip(
    y_regression - 0.55 * np.maximum(regression_severity, 0) + regression_noise,
    a_min=0.05,
    a_max=None,
)
regression_residual = y_regression - y_regression_pred

mean_baseline_prediction = np.repeat(y_regression.mean(), n_regression)
median_baseline_prediction = np.repeat(np.median(y_regression), n_regression)

pd.DataFrame({
    "actual_hours": y_regression,
    "predicted_hours": y_regression_pred,
    "residual_hours": regression_residual,
}).describe(percentiles=[0.1, 0.5, 0.9])

#### 3-4-2. MAE, median absolute error와 RMSE

MAE는 absolute error의 평균이라 target과 같은 단위로 읽을 수 있습니다. Median absolute error는 sample 절반의 absolute error가 이 값 이하라는 뜻으로 outlier의 영향을 덜 받습니다. RMSE는 큰 error를 제곱해 더 강하게 벌점하고 다시 원래 단위로 돌아옵니다.

어떤 metric이 더 “좋은” 것이 아니라 큰 error의 비용이 얼마나 빠르게 커지는지에 따라 선택합니다. 평균 하나와 함께 error quantile을 확인하면 tail risk를 더 직접적으로 볼 수 있습니다.

In [ ]:
absolute_error = np.abs(regression_residual)
regression_error_metrics = pd.Series({
    "mae_hours": mean_absolute_error(y_regression, y_regression_pred),
    "median_absolute_error_hours": median_absolute_error(y_regression, y_regression_pred),
    "mse_hours_squared": mean_squared_error(y_regression, y_regression_pred),
    "rmse_hours": root_mean_squared_error(y_regression, y_regression_pred),
    "absolute_error_p90_hours": np.quantile(absolute_error, 0.90),
    "mean_baseline_rmse_hours": root_mean_squared_error(
        y_regression,
        mean_baseline_prediction,
    ),
    "median_baseline_mae_hours": mean_absolute_error(
        y_regression,
        median_baseline_prediction,
    ),
})
regression_error_metrics.to_frame("value")

#### 3-4-3. relative error와 target scale

MAPE는 absolute percentage error의 평균이라 scale이 다른 series를 비교하기 쉬워 보이지만, actual이 0 또는 0에 가까우면 값이 폭발합니다. 음수 target에서도 해석이 어색합니다. RMSLE는 positive target에서 ratio 성격의 error를 반영하지만 0보다 작은 prediction에는 적용할 수 없습니다.

Metric이 target scale에 민감하다면 unit이 있는 MAE/RMSE와 함께 target IQR로 나눈 normalized error 등을 명시적으로 정의할 수 있습니다. 분모와 적용 가능한 target 범위를 contract에 남깁니다.

In [ ]:
target_iqr = np.quantile(y_regression, 0.75) - np.quantile(y_regression, 0.25)
regression_relative_metrics = pd.Series({
    "mape": mean_absolute_percentage_error(y_regression, y_regression_pred),
    "rmsle": root_mean_squared_log_error(y_regression, y_regression_pred),
    "mae_divided_by_target_iqr": (
        mean_absolute_error(y_regression, y_regression_pred) / target_iqr
    ),
})

near_zero_mape = mean_absolute_percentage_error(
    np.array([0.01, 10.0]),
    np.array([0.50, 9.0]),
)
print({"near_zero_example_mape": near_zero_mape})
regression_relative_metrics.to_frame("value")

#### 3-4-4. $R^2$와 explained variance

$R^2$는 constant mean baseline과 비교해 squared error가 얼마나 줄었는지 나타냅니다. 1이 최선이고, 0은 mean baseline 수준이며, 음수도 가능합니다. 음수는 “상관관계가 음수”라는 뜻이 아니라 해당 evaluation data에서 mean baseline보다 squared error가 크다는 뜻입니다.

Explained variance는 residual variance를 보지만 constant bias에는 덜 민감할 수 있습니다. $R^2$가 높아도 특정 target range나 subgroup의 error가 작다는 보장은 없으므로 absolute error와 residual을 함께 봅니다.

In [ ]:
regression_fit_metrics = pd.Series({
    "r2": r2_score(y_regression, y_regression_pred),
    "explained_variance": explained_variance_score(y_regression, y_regression_pred),
    "mean_residual_bias_hours": regression_residual.mean(),
})
regression_fit_metrics.to_frame("value")

#### 3-4-5. residual distribution과 heteroscedasticity

하나의 summary metric은 error의 방향, target range별 spread와 outlier를 숨깁니다. Actual-predicted plot에서는 identity line과의 거리를, residual plot에서는 0 주변의 pattern을 확인합니다. Prediction이 커질수록 residual spread가 넓어지는 heteroscedasticity가 보이면 overall RMSE만으로 모든 range의 품질을 설명하기 어렵습니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), layout="constrained")
fig.suptitle(f"Regression diagnostic plots (validation n={n_regression})")
plot_limit = max(y_regression.max(), y_regression_pred.max())

axes[0].scatter(
    y_regression,
    y_regression_pred,
    s=20,
    alpha=0.55,
    color=COLORS["blue"],
    edgecolor="none",
)
axes[0].plot([0, plot_limit], [0, plot_limit], color=COLORS["ink"], linestyle="--")
axes[0].set(
    title="Actual vs predicted",
    xlabel="actual hours",
    ylabel="predicted hours",
    xlim=(0, plot_limit),
    ylim=(0, plot_limit),
)

axes[1].scatter(
    y_regression_pred,
    regression_residual,
    s=20,
    alpha=0.55,
    color=COLORS["orange"],
    edgecolor="none",
)
axes[1].axhline(0, color=COLORS["ink"], linestyle="--")
axes[1].set(
    title="Residual vs predicted",
    xlabel="predicted hours",
    ylabel="actual - predicted (hours)",
)

axes[2].hist(
    regression_residual,
    bins=24,
    color=COLORS["blue_light"],
    edgecolor=COLORS["blue"],
)
axes[2].axvline(0, color=COLORS["ink"], linestyle="--")
axes[2].set(
    title="Residual distribution",
    xlabel="actual - predicted (hours)",
    ylabel="samples",
)
plt.show()

#### 3-4-6. asymmetric cost와 pinball loss

Underprediction과 overprediction의 비용이 다르면 symmetric MAE나 RMSE만으로 목적을 표현하기 어렵습니다. Pinball loss의 `alpha`는 어떤 conditional quantile을 목표로 하는지 나타냅니다. `alpha=0.5`는 median을 목표로 하며 MAE와 연결되고, 높은 alpha는 underprediction에 더 큰 penalty를 줍니다.

서로 다른 alpha의 prediction을 비교할 때는 각 model이 해당 quantile을 예측하도록 학습되었는지 확인해야 합니다. 아래 계산은 같은 point prediction에 cost 관점만 달리 적용하는 API 예제입니다.

In [ ]:
pinball_metrics = pd.Series({
    "pinball_alpha_0_50": mean_pinball_loss(
        y_regression,
        y_regression_pred,
        alpha=0.50,
    ),
    "pinball_alpha_0_80": mean_pinball_loss(
        y_regression,
        y_regression_pred,
        alpha=0.80,
    ),
    "pinball_alpha_0_95": mean_pinball_loss(
        y_regression,
        y_regression_pred,
        alpha=0.95,
    ),
})
pinball_metrics.to_frame("loss")

#### 3-4-7. target range와 subgroup slice

Overall regression metric이 좋아도 긴 stay나 특정 cohort에서 큰 error가 날 수 있습니다. Target range별 sample 수, MAE, bias와 p90 absolute error를 함께 계산합니다. Data가 적은 slice는 metric 변동이 크므로 count와 interval 없이 순위를 매기지 않습니다.

In [ ]:
regression_slice_frame = pd.DataFrame({
    "actual": y_regression,
    "predicted": y_regression_pred,
    "residual": regression_residual,
    "absolute_error": absolute_error,
})
regression_slice_frame["target_band"] = pd.qcut(
    regression_slice_frame["actual"],
    q=[0, 0.5, 0.8, 1.0],
    labels=["lower 50%", "50-80%", "upper 20%"],
)

regression_slice_metrics = (
    regression_slice_frame.groupby("target_band", observed=True)
    .agg(
        samples=("actual", "size"),
        mae_hours=("absolute_error", "mean"),
        bias_hours=("residual", "mean"),
        absolute_error_p90_hours=("absolute_error", lambda values: values.quantile(0.90)),
    )
)
regression_slice_metrics

### 3-5. Metric을 비교 가능한 evidence로 만들기

#### 3-5-1. bootstrap interval로 sample uncertainty 확인하기

같은 model과 metric도 evaluation sample이 달라지면 값이 달라집니다. Nonparametric bootstrap은 validation row를 replacement와 함께 다시 뽑아 metric 분포를 근사합니다. 아래 interval은 교육용 percentile interval입니다. Patient의 여러 row가 섞인 data라면 row가 아니라 patient나 encounter 같은 독립 단위를 resampling해야 합니다.

Interval은 data shift, label error, 여러 번의 model selection에서 생기는 uncertainty를 모두 해결하지 않습니다. Split strategy와 독립 단위를 먼저 맞춘 뒤 보조 evidence로 사용합니다.

In [ ]:
def bootstrap_interval(metric_function, *arrays, repeats=300, seed=SEED):
    '''Return a percentile interval by resampling aligned observations.'''
    local_rng = np.random.default_rng(seed)
    sample_count = len(arrays[0])
    estimates = []
    for _ in range(repeats):
        indices = local_rng.integers(0, sample_count, size=sample_count)
        estimates.append(metric_function(*(array[indices] for array in arrays)))
    return np.quantile(estimates, [0.025, 0.5, 0.975])


binary_auc_interval = bootstrap_interval(roc_auc_score, y_binary, p_binary)
multiclass_f1_interval = bootstrap_interval(
    lambda actual, predicted: f1_score(actual, predicted, average="macro"),
    y_multiclass,
    y_multiclass_pred,
)
regression_mae_interval = bootstrap_interval(
    mean_absolute_error,
    y_regression,
    y_regression_pred,
)

bootstrap_summary = pd.DataFrame(
    [binary_auc_interval, multiclass_f1_interval, regression_mae_interval],
    index=["binary AUROC", "multiclass macro F1", "regression MAE (hours)"],
    columns=["p2.5", "median", "p97.5"],
)
bootstrap_summary

#### 3-5-2. model selection metric과 reporting metric

Model selection에서는 primary scorer 하나와 tie-break·guardrail 규칙을 미리 정합니다. Reporting에서는 primary뿐 아니라 baseline, class/slice, uncertainty와 threshold를 함께 남깁니다. Cross-validation의 fold별 score를 평균 하나로 지우지 말고 spread와 split definition을 보존합니다.

Loss를 `scoring=`에 넘길 때 scikit-learn의 `neg_` scorer는 부호를 바꾼 값이라는 점에 주의합니다. 예를 들어 `neg_mean_absolute_error=-2.3`은 보고할 MAE가 `2.3`이라는 뜻입니다.

In [ ]:
evaluation_evidence = pd.DataFrame(
    [
        {
            "task": "binary",
            "primary_metric": "AUROC",
            "primary_value": binary_ranking_metrics["auroc"],
            "guardrail": "recall at threshold",
            "guardrail_value": binary_label_metrics["recall_sensitivity"],
            "baseline_metric": "random ranking AUROC",
            "baseline_value": binary_baseline_metrics["random_ranking_auroc"],
        },
        {
            "task": "multiclass",
            "primary_metric": "macro F1",
            "primary_value": multiclass_summary["f1_macro"],
            "guardrail": "severe-class recall",
            "guardrail_value": multiclass_report.loc["severe", "recall"],
            "baseline_metric": "majority macro F1",
            "baseline_value": multiclass_baseline_metrics["majority_macro_f1"],
        },
        {
            "task": "regression",
            "primary_metric": "MAE (hours)",
            "primary_value": regression_error_metrics["mae_hours"],
            "guardrail": "p90 absolute error",
            "guardrail_value": regression_error_metrics["absolute_error_p90_hours"],
            "baseline_metric": "median baseline MAE",
            "baseline_value": regression_error_metrics["median_baseline_mae_hours"],
        },
    ]
)
evaluation_evidence

#### 3-5-3. 흔한 해석 오류 점검

- Accuracy 하나로 imbalanced classification을 결론내리지 않습니다.
- ROC AUC가 높다는 이유로 probability가 calibrated되었다고 쓰지 않습니다.
- Multiclass 평균에는 `macro`, `micro`, `weighted` 중 무엇을 썼는지 표시합니다.
- Top-k accuracy를 최종 single-class decision의 accuracy처럼 보고하지 않습니다.
- MAE와 RMSE에는 target unit을 붙이고, 서로 다른 scale의 target을 그대로 비교하지 않습니다.
- $R^2$가 음수일 수 있음을 오류로 오해하지 않고 baseline보다 나쁜 squared error인지 확인합니다.
- Threshold, class order, positive label, missing row 처리와 sample weight를 metric key 밖의 숨은 설정으로 남기지 않습니다.
- 같은 validation data를 반복해서 보며 metric을 최적화했다면 selection bias 가능성을 기록합니다.

## 4. Checks

Notebook을 마치기 전에 prediction shape, probability 범위, metric direction과 baseline 비교를 확인합니다. 이 check는 실제 project의 release gate를 대신하지 않지만 예제 계산이 의도한 contract를 따르는지 검증합니다.

In [ ]:
assert y_binary.shape == p_binary.shape
assert set(np.unique(y_binary)) <= {0, 1}
assert np.all((0 <= p_binary) & (p_binary <= 1))
assert binary_ranking_metrics["auroc"] > 0.5
assert np.isclose(
    binary_ranking_metrics["auroc"],
    binary_ranking_metrics["auroc_from_curve"],
)
assert binary_ranking_metrics["pr_auc_trapezoid"] >= 0
assert binary_probability_metrics["log_loss"] >= 0

assert p_multiclass.shape == (n_multiclass, len(class_labels))
assert np.allclose(p_multiclass.sum(axis=1), 1.0)
assert set(np.unique(y_multiclass)) == set(class_labels)
assert np.isclose(multiclass_summary["f1_micro"], multiclass_summary["accuracy"])
assert 0 <= multiclass_probability_metrics["top_2_accuracy"] <= 1

assert y_regression.shape == y_regression_pred.shape
assert np.all(y_regression >= 0)
assert np.all(y_regression_pred >= 0)
assert regression_error_metrics["mae_hours"] >= 0
assert regression_error_metrics["rmse_hours"] >= regression_error_metrics["mae_hours"]
assert regression_error_metrics["mae_hours"] < regression_error_metrics["median_baseline_mae_hours"]

assert list(bootstrap_summary.columns) == ["p2.5", "median", "p97.5"]
assert (bootstrap_summary["p2.5"] <= bootstrap_summary["median"]).all()
assert (bootstrap_summary["median"] <= bootstrap_summary["p97.5"]).all()

print("All appendix checks passed.")

## 5. Next Steps

- 실제 model에서는 validation prediction을 row-level evidence로 versioning하고 metric을 다시 계산할 수 있게 보존합니다.
- Metric contract에 target revision, split, class order, threshold, averaging, sample weight, unit과 direction을 포함합니다.
- Binary는 threshold·ranking·calibration을, multiclass는 per-class·averaging·probability를, regression은 scale·tail error·residual을 함께 검토합니다.
- Group별 metric은 sample 수와 interval을 함께 보고하고, 작은 group의 순위를 과도하게 해석하지 않습니다.
- 다음 MLflow notebook에서는 여기서 정의한 metric key와 evaluation artifact를 Run에 기록합니다.

## 6. References

- [scikit-learn: Metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn API: `sklearn.metrics`](https://scikit-learn.org/stable/api/sklearn.metrics.html)
- [scikit-learn API: `auc`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.auc.html)
- [scikit-learn API: `average_precision_score`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)
- [scikit-learn: Probability calibration](https://scikit-learn.org/stable/modules/calibration.html)
- [scikit-learn API: `calibration_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html)
- [scikit-learn API: `mean_pinball_loss`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_pinball_loss.html)